Import Libraries

In [14]:
import pandas as pd
import os
from skimage.transform import resize
from skimage.io import imread
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [15]:
import os
print("WORKING DIRECTORY:", os.getcwd())
print("CONTENTS:", os.listdir())
print(os.listdir("Nums"))




WORKING DIRECTORY: c:\Users\cdwit\OneDrive\Documents\GitHub\486-Final-Project\Code\SVM
CONTENTS: ['cwitte_SVM.ipynb', 'Nums']
['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']


Load Images and Convert to Data Frame

In [16]:
from skimage.transform import rotate
from skimage.util import random_noise

def augment(img):
    imgs = []
    imgs.append(rotate(img, angle=np.random.uniform(-10, 10)))
    imgs.append(np.roll(img, shift=np.random.randint(-2, 2), axis=0))
    imgs.append(np.roll(img, shift=np.random.randint(-2, 2), axis=1))
    imgs.append(random_noise(img, var=0.01))
    return imgs


categories = [str(i) for i in range(10)]
flat_data_arr = [] # Input array
target_arr = [] # Output array
datadir = "Nums" # Path containing number images

for i in categories:
    print(f"Loading category: {i}")
    path = os.path.join(datadir, i)
    
    for img in os.listdir(path):
        img_array = imread(os.path.join(path, img), as_gray=True)
        img_resized = resize(img_array,(28, 28))
        flat_data_arr.append(img_resized.flatten())
        target_arr.append(int(i))

        augmented_img = augment(img_resized)

        for aug in augmented_img:
            flat_data_arr.append(aug.flatten())
            target_arr.append(int(i))
    
    print(f"Loaded category {i} successfully")

flat_data = np.array(flat_data_arr)
target = np.array(target_arr)

# Create Data Frame
df = pd.DataFrame(flat_data)
df["Target"] = target


Loading category: 0
Loaded category 0 successfully
Loading category: 1
Loaded category 1 successfully
Loading category: 2
Loaded category 2 successfully
Loading category: 3
Loaded category 3 successfully
Loading category: 4
Loaded category 4 successfully
Loading category: 5
Loaded category 5 successfully
Loading category: 6
Loaded category 6 successfully
Loading category: 7
Loaded category 7 successfully
Loading category: 8
Loaded category 8 successfully
Loading category: 9
Loaded category 9 successfully


Separate Input Features and Targets

In [17]:
x = df.iloc[:, :-1] # Input data
y = df.iloc[:, -1] # Output data

# Split data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.50, random_state=42, stratify=y)

Build and Train Model

In [18]:
# Define parameters grid for GridSearchCV
param_grid = {"C": [0.1, 1, 10, 100], "gamma": [0.0001, 0.001, 0.1, 1], "kernel": ["rbf", "poly"]}

# Create support vector classifier
svc = svm.SVC(probability=True)

# Create a model using GridSearchCV with the parameters grid
model = GridSearchCV(svc, param_grid, cv=2)

# Traininng model
model.fit(x_train, y_train)

GridSearchCV(cv=2, estimator=SVC(probability=True),
             param_grid={'C': [0.1, 1, 10, 100],
                         'gamma': [0.0001, 0.001, 0.1, 1],
                         'kernel': ['rbf', 'poly']})

Model Evaluation

In [19]:
# Testing model
y_pred = model.predict(x_test)

# Calculating the accuracy
accuracy = accuracy_score(y_test, y_pred)

# Print accuracy
print(f"The model is {accuracy*100:.2f}% accurate")

# Classification report
print(classification_report(y_test, y_pred, target_names=categories))


The model is 90.67% accurate
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         7
           1       0.70      1.00      0.82         7
           2       1.00      0.75      0.86         8
           3       0.78      0.88      0.82         8
           4       1.00      0.88      0.93         8
           5       1.00      0.71      0.83         7
           6       1.00      1.00      1.00         8
           7       1.00      0.86      0.92         7
           8       1.00      1.00      1.00         7
           9       0.80      1.00      0.89         8

    accuracy                           0.91        75
   macro avg       0.93      0.91      0.91        75
weighted avg       0.93      0.91      0.91        75

